AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 06. 회전 각도별 CNN 학습하고 저장하기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- 회전 범위만 변경하는 비교 실험을 반복문으로 구성할 수 있다.
- 시드·데이터 분할·모델 구조 등 실험 조건을 동일하게 유지할 수 있다.
- 학습한 모델과 에포크별 학습 기록을 파일로 저장하고, 학습과 평가를 구분할 수 있다.

### VS Code 실행 준비

`DATA_ROOT`를 4강 이미지 폴더에 맞추세요. 학습 모델과 에포크별 기록은 `models`에 저장합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers

COURSE_DIR = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_05강')
DATA_ROOT = COURSE_DIR.parent / "wafermap_image_data_261021"

IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]
BATCH_SIZE = 32
EPOCHS = 10
TRAIN_DIR = DATA_ROOT / "Train"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def check_image_folders(directory):
    missing = [
        name for name in class_names
        if not any(
            path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            for path in (directory / name).rglob("*")
        )
    ]
    if missing:
        raise FileNotFoundError(
            f"{directory}에 이미지가 없는 클래스: {', '.join(missing)}. "
            "첫 셀의 DATA_ROOT를 4강 Train/Test가 있는 폴더로 설정하세요."
        )

In [ ]:
SEED = 42

import numpy as np
import tensorflow as tf

def list_images(directory):
    check_image_folders(directory)
    image_paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_paths = sorted(
            path for path in (directory / class_name).rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        image_paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return image_paths, np.asarray(labels, dtype=np.int32)

def decode_image(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

def make_dataset(image_paths, labels, shuffle=False, seed=42):
    # Python이 파일을 읽어 Windows의 한글 경로를 처리합니다.
    image_bytes = [path.read_bytes() for path in image_paths]
    dataset = tf.data.Dataset.from_tensor_slices((image_bytes, labels))
    if shuffle:
        dataset = dataset.shuffle(BATCH_SIZE * 8, seed=seed)
    dataset = dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def create_experiment_datasets():
    image_paths, labels = list_images(TRAIN_DIR)

    # 같은 파일 목록과 시드로 학습·검증을 80:20으로 나눕니다.
    indices = np.random.RandomState(SEED).permutation(len(image_paths))
    val_count = int(len(indices) * 0.2)
    train_indices, val_indices = indices[:-val_count], indices[-val_count:]

    train_exp = make_dataset(
        [image_paths[i] for i in train_indices], labels[train_indices],
        shuffle=True, seed=SEED,
    )
    val_exp = make_dataset([image_paths[i] for i in val_indices], labels[val_indices])
    return train_exp, val_exp

def create_cnn_model(rotation_degree=0):
    # 모델 초기화 전에 난수 시드 고정
    keras.utils.set_random_seed(SEED)

    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.RandomRotation(
            factor=rotation_degree / 360,
            fill_mode="constant",
            fill_value=0.0,
            interpolation="nearest"
        ),
        layers.Rescaling(1./255),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(len(class_names), activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## 1. 이번 실험에서는 무엇을 바꿀까?

Mission 05에서 `create_cnn_model()`과 `create_experiment_datasets()`를 만들었다. 이제 이 함수를 사용해 **회전 각도만 바꾸면서 CNN을 각각 처음부터 학습**해보자.

| 실험 | 회전 범위 | 함수 전달값 | 저장 파일명 |
| --- | --- | --- | --- |
| A | 회전 없음 | `0` | `cnn_rotation_0.keras` |
| B | ±15° | `15` | `cnn_rotation_15.keras` |
| C | ±45° | `45` | `cnn_rotation_45.keras` |
| D | ±90° | `90` | `cnn_rotation_90.keras` |

네 실험에서 CNN 구조, 데이터 분할, 에포크, 배치 크기, 시드는 가능한 한 동일하게 유지한다. **이 실험의 유일한 의도적 변경 변수는 회전 범위**다.

회전 각도 리스트 [0, 15, 45, 90]하나씩 반복

↓

데이터셋 준비 → 새 CNN 생성 → 학습동일한 시드와 학습 설정

↓

모델 파일 + 학습 기록 저장다음 회전 각도로 진행

회전이 없는 모델도 `RandomRotation(factor=0)`를 포함하지만 실제 이미지 회전은 발생하지 않는다. 실험의 기준 모델로 사용할 수 있다.

## 2. 실험 파일을 저장할 위치 정하기
Colab의 임시 저장 공간은 런타임이 초기화되면 사라질 수 있다. 따라서 **학습이 끝난 모델은 Google Drive에 저장**하자. 이미 Drive를 연결했다면 아래 코드에서 경로만 자신의 폴더에 맞게 수정한다.

In [ ]:
MODEL_DIR = COURSE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("저장 폴더:", MODEL_DIR)

os.makedirs(MODEL_DIR, exist_ok=True)는 저장 폴더가 없으면 만들고, 이미 있으면 그대로 사용한다. MODEL_DIR은 문자열이므로 끝에 /를 포함한다.

**확인:** Colab에서 Drive를 아직 마운트하지 않았다면 먼저 `from google.colab import drive`와 `drive.mount('/content/drive')`를 실행해야 한다. 로컬 PC라면 `MODEL_DIR`을 자신의 저장 경로로 바꾸면 된다.

## 3. 반복 실험 코드 구성하기
회전 각도가 네 개이므로 같은 코드를 네 번 복사하는 대신 `for` 반복문을 사용한다. Mission 05에서 작성한 두 함수와 `SEED`, `EPOCHS` 등의 상수를 그대로 활용하자.
**코드 작성 미션**

1. 회전 각도 네 개를 리스트에 저장한다.
2. 반복문에서 매 실험 시작 시 시드를 재설정한다.
3. 실험용 데이터셋과 새 CNN 모델을 생성한다.
4. `fit()`으로 학습한 뒤 모델을 해당 각도 이름으로 저장한다.

In [ ]:
rotation_degrees = [0, 15, 45, 90]

In [ ]:
import pandas as pd

for degree in rotation_degrees:
    print(f"\n===== 회전 ±{degree}° 실험 =====")

    # 1. 실험 시작 시 난수 시드 고정
    keras.utils.set_random_seed(SEED)

    # 2. 동일한 조건으로 데이터셋 준비
    train_exp, val_exp = create_experiment_datasets()

    # 3. 각도에 맞는 새 CNN 모델 생성
    model_exp = create_cnn_model(degree)

    # 4. 훈련 데이터로 학습하고 검증 데이터로 확인
    history_exp = model_exp.fit(
        train_exp,
        validation_data=val_exp,
        epochs=EPOCHS
    )

    # 5. 모델 저장: 경로와 파일명 연결
    filename = MODEL_DIR / f"cnn_rotation_{degree}.keras"
    model_exp.save(filename)
    print("모델 저장:", filename)

    # 에포크별 학습/검증 지표를 표로 변환
    history_df = pd.DataFrame(history_exp.history)
    history_file = MODEL_DIR / f"cnn_rotation_{degree}_history.csv"
    history_df.to_csv(history_file, index=False)
    print("학습 기록 저장:", history_file)

여기서 `create_cnn_model()`은 호출할 때마다 시드를 다시 설정하도록 만든 함수이므로 모델 초기화 조건을 최대한 동일하게 맞출 수 있다. 반복문 맨 앞의 시드 설정은 데이터셋을 만드는 과정까지 함께 같은 출발점으로 맞추기 위한 것이다.

학습은 각 모델마다 `EPOCHS`만큼 진행된다. 네 모델을 각각 학습하므로 전체 실행 시간은 길어질 수 있다.

**실행 전 확인:** Mission 05에서 `create_cnn_model()`의 CNN 레이어를 완성했는지, `class_names`와 `TRAIN_DIR` 등이 준비되어 있는지 확인하자.

## 4. 학습 기록(History)도 함께 저장하기
`fit()`의 반환값인 `history_exp`에는 에포크별 학습 정확도와 손실, 검증 정확도와 손실이 기록된다. 그러나 반복문을 한 번 더 돌면 이 변수에는 **마지막 실험의 History만 남는다.**
또한 `.keras` 파일을 저장하더라도 `fit()`이 반환한 과거 에포크별 기록 전체가 자동으로 남지는 않는다. 실험 결과를 나중에 비교할 수 있도록 CSV로 따로 저장하자.

위 학습 셀의 반복문 안에 학습 기록을 저장하는 코드가 포함되어 있습니다.

이 코드는 **3번 반복문 안에서 모델을 저장한 직후**에 추가한다. 새 셀에 들여쓰기된 코드만 따로 실행하는 것이 아니다.

| 파일 | 저장하는 것 |
| --- | --- |
| `cnn_rotation_45.keras` | 학습한 모델과 가중치 |
| `cnn_rotation_45_history.csv` | 에포크별 accuracy, loss, val_accuracy, val_loss |

History 파일은 별도의 재학습 없이 학습 곡선을 그리는 데 사용할 수 있다.

## 5. 학습과 결과 분석을 분리하는 이유
모델을 학습하는 코드와 결과를 비교하는 코드를 분리하면 편리하다. 예를 들어 Colab 연결이 중간에 끊기더라도 **이미 저장된 모델을 다시 학습할 필요 없이 불러와 평가**할 수 있다.
**학습 단계**

새 모델 생성

fit()으로 학습

save()로 파일 저장

시간이 많이 걸리므로 완료한 모델부터 보관한다.

**분석 단계**

저장 파일 불러오기

evaluate()로 평가

결과표 · 혼동행렬 작성

필요할 때 반복 실행해 결과를 확인할 수 있다.
이번 미션의 중심 작업은 **학습하고 파일로 저장하는 것**이다. 성능지표를 한 번에 비교하는 코드는 다음 분석 단계에서 작성한다.

In [ ]:
# 문자열 경로의 존재 여부는 os.path.exists()로 확인한다.
import os

for degree in rotation_degrees:
    filename = MODEL_DIR / f"cnn_rotation_{degree}.keras"
    print(degree, "° :", "저장 완료" if os.path.exists(filename) else "파일 없음")

위 코드는 네 실험이 전부 끝난 뒤 실행한다. 학습이 중간에 멈췄다면 저장이 끝나지 않은 각도만 선택해 다시 실행할 수 있다.

아래 예시처럼 각도 목록 셀을 수정한 뒤, 그 셀과 학습 셀을 다시 실행하세요.

```python
# 예: 90° 모델만 저장하지 못했을 때
rotation_degrees = [90]

# 위의 학습 반복문을 다시 실행한다.
```

파일이 존재하는지만 확인한 것이므로 모델의 정상 로드 여부까지 보장하지는 않는다. 중요한 모델은 이후 `keras.models.load_model()`로 불러와 확인할 수 있다. 파일명이 같으면 저장 시 기존 파일이 갱신되므로, 재학습 시에는 필요한 파일을 덮어쓰는지 확인하자.

## 6. 코드 이해 및 실험 설계 활동

**Q6.1.**

`rotation_degrees = [0, 15, 45, 90]`에서 숫자들은 인덱스인가, 함수에 전달할 회전 각도인가? `for`문은 총 몇 번 실행되는가?

**답변**

`0`, `15`, `45`, `90`은 함수에 전달하는 **회전 각도**입니다. 반복문은 각도마다 한 번씩, 총 **네 번** 실행됩니다.

**Q6.2.**

반복문에서 시드를 다시 설정하고 `create_cnn_model(degree)`로 새 모델을 만드는 이유를 설명하시오. 기존 모델에 `fit()`을 이어서 실행하는 것과 무엇이 다른가?

**답변**

같은 시드에서 각 회전 조건의 모델을 처음부터 학습하기 위해서입니다. 기존 모델에 `fit()`을 이어서 실행하면 앞서 학습한 가중치에서 시작하므로 추가 학습의 영향이 섞입니다.

**Q6.3.**

`filename = MODEL_DIR + f"cnn_rotation_{degree}.keras"`에서 degree가 45일 때 저장되는 파일명은 무엇인가?

**답변**

파일명은 **`cnn_rotation_45.keras`**이며, `MODEL_DIR`에 저장됩니다.

**Q6.4.**

`model_exp.save()`와 `history_df.to_csv()`는 각각 무엇을 저장하는가? 둘 중 하나만 저장한다면 어떤 정보를 잃게 되는가?

**답변**

`.keras` 파일은 **모델 구조와 가중치**를, History CSV는 **에포크별 학습·검증 정확도와 손실**을 저장합니다. 모델만 있으면 학습 곡선의 기록이 없고, CSV만 있으면 모델을 복원해 예측할 수 없습니다.

**Q6.5.**

네 모델 중 ±90° 학습 도중 연결이 끊겼다. 나머지 세 모델이 정상 저장되어 있다면 코드를 어떻게 수정해 필요한 모델만 학습할 수 있는가?

**답변**

`rotation_degrees = [90]`으로 바꾼 후 각도 목록 셀과 학습 셀을 다시 실행합니다. 그러면 ±90° 모델만 학습합니다.

**Q6.6.**

이번 실험에서는 무엇을 바꾸고 어떤 조건을 동일하게 유지했는가? 실험 결과가 다른 이유를 회전 각도 때문이라고 단정하기 어려운 이유도 한 가지 작성하시오.

**답변**

**회전 범위**를 바꾸고 시드, 데이터 분할, CNN 구조, 배치 크기, 에포크 수를 고정했습니다. 같은 시드라도 실행 환경에 따른 연산의 변동이 생길 수 있어 반복 실험으로 확인해야 합니다.